# DX799 Week 10: K-Means Clustering, Elbow Method, Silhouette Scores, and Distance Metrics
**Integrated Capstone Project:** VFX, Animation, and the Film Industry  
**Research question:** Can I identify meaningful groups of movies based on information available before release, and do those groups differ in their observed box-office outcomes?

In Weeks 8 and 9, I used supervised models to predict whether a movie earned at least twice its production budget. This week I am shifting to **unsupervised learning**. K-Means will group movies using their pre-release characteristics **without seeing their revenue or success status**. I will use the elbow method and silhouette scores to decide whether the groups are meaningful, check how scaling affects them, and then examine movie revenue outcomes **after** the clusters have been formed. The goal is to find useful patterns, not claim that belonging to a cluster causes success.

## 1. Method and safeguards
K-Means repeatedly assigns movies to the closest cluster center and recomputes those centers to reduce **within-cluster squared Euclidean distance**, also called inertia. The chosen number of clusters, **k**, controls how many groups the algorithm creates. A very small k can combine unlike movies; a large k may split movies into groups without making the patterns easier to interpret.

- **Elbow method:** Plot inertia for different k values. Inertia always falls as k increases, so I will look for where the improvement starts to slow down instead of choosing the lowest inertia.
- **Silhouette score:** Compare how close movies are to their assigned group versus other groups. Scores near 1 indicate better separation; values near 0 suggest overlapping groups. Negative scores suggest a movie may fit another cluster better.
- **Scaling:** Budget (in millions), release year, runtime, binary genres, and month indicators use different scales. I will compare standardized and robust-scaled inputs, as well as an intentionally unscaled baseline.
- **Distance metrics:** Standard scikit-learn K-Means optimizes **squared Euclidean** distance, not Manhattan distance. I will show how Euclidean and Manhattan silhouette evaluations differ for a *fixed* clustering. This is a diagnostic, not a claim that K-Means optimized Manhattan distance.

I will fit the unsupervised clusters on the same training subset used earlier and inspect cluster quality there. For evaluation, the reserved test movies will only be assigned to the final centers. I will never include `revenue`, `gross_to_budget`, or `success` among the clustering predictors. The sample's positive-revenue filter remains a limitation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score
from sklearn.decomposition import PCA

SEED = 42
DATA_PATH = Path('movies.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('/mnt/data/movies.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Place the same movies.csv from Weeks 8 and 9 beside this notebook, or update DATA_PATH.')
raw = pd.read_csv(DATA_PATH)
print('Input file:', DATA_PATH)
print('Original data:', raw.shape)

## 2. Prepare a consistent movie sample
I am using the same selection criteria and 19 pre-release variables as my Week 8 and Week 9 notebooks. I remove duplicate movie IDs, keep releases from 1980 to 2024, require positive reported budgets and revenues, and keep runtimes of 40–240 minutes. The last two requirements make the sample less representative of movies with missing financial data. The two-times-budget success flag is computed **only to describe clusters later**, not to create them.

I keep the 80/20 random split (`random_state=42`) for consistency with earlier weeks. Because this is clustering rather than prediction, I do not train a classifier or use the success label to choose k.

In [ ]:
df = raw.drop_duplicates(subset=['movie_id']).copy()
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = pd.to_numeric(df['release_year'], errors='coerce')
for column in ['budget', 'revenue', 'runtime']:
    df[column] = pd.to_numeric(df[column], errors='coerce')
df = df.loc[(df.budget > 0) & (df.revenue > 0) &
            df.release_year.between(1980, 2024) &
            df.runtime.between(40, 240) & df.release_date.notna()].copy()
df['budget_m'] = df.budget / 1e6
df['revenue_m'] = df.revenue / 1e6
df['gross_to_budget'] = df.revenue / df.budget
df['success'] = (df.gross_to_budget >= 2).astype(int)
genres = df.genres.fillna('').str.split('|')
df['animation'] = genres.map(lambda g: int('Animation' in g))
df['release_month'] = df.release_date.dt.month
for genre in ['Action', 'Comedy', 'Drama', 'Horror']:
    df['genre_' + genre.lower()] = genres.map(lambda g: int(genre in g))
for month in range(2, 13):
    df[f'month_{month}'] = (df.release_month == month).astype(int)
features = ['budget_m','runtime','release_year','animation',
            'genre_action','genre_comedy','genre_drama','genre_horror'] + [f'month_{i}' for i in range(2, 13)]
tr, te = train_test_split(np.arange(len(df)), test_size=.2, random_state=SEED)
train, test = df.iloc[tr].copy(), df.iloc[te].copy()
X_train = train[features].copy()
X_test = test[features].copy()
assert X_train.notna().all().all() and X_test.notna().all().all()
print(f'Filtered movies: {len(df):,}; training: {len(train):,}; test: {len(test):,}')
print(f'Predictors: {len(features)}; animated films: {df.animation.sum():,}')
print(f'Overall success rate (not used to fit clusters): {df.success.mean():.1%}')
print('Features:', features)

## 3. Exploratory Data Analysis and scaling
Budget is usually highly skewed, and many of my predictors are binary indicators. I will look at the budget distribution and the relative variation in the numerical features before choosing a scaling method. StandardScaler gives each column a mean of zero and unit standard deviation. RobustScaler centers by the median and scales by the interquartile range, so it is less affected by extreme observations. Both are fitted on **training data only** and then applied to test movies.

A caution: scaling every individual month dummy to unit variance can unintentionally give the 11 calendar indicators substantial combined influence. I will therefore also test a compact representation with **cyclical release month** rather than the 11 month indicators. This is a sensitivity analysis, not a replacement of the original Week 8–9 19-predictor design.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(train['budget_m'].clip(upper=250), bins=35)
axes[0].set(title='Training movie budgets (display capped at $250M)',
            xlabel='Budget ($ millions)', ylabel='Movies')
axes[1].scatter(train['budget_m'].clip(upper=250), train['runtime'], alpha=.18, s=10)
axes[1].set(title='Budget and runtime: training sample', xlabel='Budget ($M; clipped)', ylabel='Runtime (minutes)')
plt.tight_layout(); plt.show()
print(train[['budget_m', 'runtime', 'release_year', 'revenue_m']].describe().round(2).to_string())

In [ ]:
scalers = {'Unscaled': None, 'StandardScaler': StandardScaler(), 'RobustScaler': RobustScaler()}
scaled_train, scaled_test = {}, {}
for label, scaler in scalers.items():
    if scaler is None:
        scaled_train[label] = X_train.to_numpy(dtype=float)
        scaled_test[label] = X_test.to_numpy(dtype=float)
    else:
        scaled_train[label] = scaler.fit_transform(X_train)
        scaled_test[label] = scaler.transform(X_test)
print('Scaling fitted on training data only; test transformation reused training parameters.')

## 4. Elbow method and silhouette scores
I will compare k from 2 through 9 for each scaling option, with multiple initializations (`n_init=10`) and the same random seed. This limits instability from poor starting centroids but does not guarantee the globally best K-Means solution.

Because silhouette calculations require many pairwise distances, I use a reproducible subsample of 1,200 training movies to keep the notebook practical. The **inertia** still uses the full training sample. I will select the combination with the largest mean sampled Euclidean silhouette score, while also examining whether the clusters are extremely unbalanced and whether their profiles make sense. Silhouette is an internal measure of geometric separation, not proof of business usefulness.

In [ ]:
K_VALUES = range(2, 10)
rows = []
models = {}
for scaler_name, X in scaled_train.items():
    for k in K_VALUES:
        km = KMeans(n_clusters=k, n_init=10, random_state=SEED)
        labels = km.fit_predict(X)
        score = silhouette_score(X, labels, metric='euclidean', sample_size=1200, random_state=SEED)
        sizes = np.bincount(labels, minlength=k)
        rows.append({'Scaling': scaler_name, 'k': k, 'Inertia': km.inertia_,
                     'Silhouette (Euclidean)': score,
                     'Smallest cluster': sizes.min(), 'Largest cluster share': sizes.max()/len(labels)})
        models[(scaler_name, k)] = km
results = pd.DataFrame(rows)
print(results.round(4).to_string(index=False))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
for name in scalers:
    r = results[results['Scaling'] == name]
    axs[0].plot(r['k'], r['Inertia'], marker='o', label=name)
    axs[1].plot(r['k'], r['Silhouette (Euclidean)'], marker='o', label=name)
axs[0].set(title='Elbow method: inertia by k', xlabel='Number of clusters (k)', ylabel='Inertia')
axs[1].set(title='Sampled Euclidean silhouette by k', xlabel='Number of clusters (k)', ylabel='Silhouette score')
for ax in axs:
    ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

best_row = results.sort_values('Silhouette (Euclidean)', ascending=False).iloc[0]
best_scaling, best_k = best_row['Scaling'], int(best_row['k'])
best_model = models[(best_scaling, best_k)]
print(f'Highest sampled silhouette: {best_row["Silhouette (Euclidean)"]:.4f}; '
      f'scaling={best_scaling}, k={best_k}')
print('Cluster sizes:', np.bincount(best_model.labels_, minlength=best_k))
print('Important: compare cluster balance and business interpretation before accepting this as useful.')

## 5. Compare Euclidean and Manhattan distance metrics
There is an important distinction between **training distance** and **evaluation distance**. Ordinary K-Means always locates means to minimize squared Euclidean distances. A Manhattan silhouette score can still evaluate the assignments from that same fitted K-Means model, but it does not turn the algorithm into Manhattan K-Means.

I compare both silhouette metrics for the selected clustering to see whether the appearance of separation depends on how distance is measured. I also show a small table of Manhattan versus Euclidean **distances to cluster centers** for a few movies. For a true Manhattan-based clustering algorithm, K-Medians or K-Medoids would be more appropriate and would require a separate implementation.

In [ ]:
from scipy.spatial.distance import cdist
X_final = scaled_train[best_scaling]
labels = best_model.labels_
sil_eu = silhouette_score(X_final, labels, metric='euclidean', sample_size=1200, random_state=SEED)
sil_ma = silhouette_score(X_final, labels, metric='manhattan', sample_size=1200, random_state=SEED)
print(pd.DataFrame({'Evaluation distance': ['Euclidean', 'Manhattan'],
                    'Silhouette': [sil_eu, sil_ma]}).round(4).to_string(index=False))
center_distance_eu = cdist(X_final[:5], best_model.cluster_centers_, metric='euclidean')
center_distance_ma = cdist(X_final[:5], best_model.cluster_centers_, metric='cityblock')
print('First five training movies: closest Euclidean center:', center_distance_eu.argmin(axis=1))
print('First five training movies: closest Manhattan center:', center_distance_ma.argmin(axis=1))
print('K-Means assignments continue to use Euclidean distance, regardless of Manhattan diagnostic.')

## 6. Sensitivity check: avoid overweighting month dummy variables
The original 19 features are useful for comparability with Weeks 8–9, but K-Means can be sensitive to the representation of categorical variables. I will compare the original 19-feature input with a compact 10-variable design containing a continuous budget, runtime and year, five genre flags, plus `sin` and `cos` of the release month. Sin/cos represent December and January as nearby times of year rather than opposite ends of a 1–12 line.

This test helps establish whether the resulting clusters reflect meaningful movie differences or simply the prevalence and weighting of calendar indicators. It is a sensitivity test using training data only.

In [ ]:
compact_train = train[['budget_m','runtime','release_year','animation',
    'genre_action','genre_comedy','genre_drama','genre_horror']].copy()
compact_train['month_sin'] = np.sin(2*np.pi*train.release_month/12)
compact_train['month_cos'] = np.cos(2*np.pi*train.release_month/12)
compact_scaler = StandardScaler()
compact_X = compact_scaler.fit_transform(compact_train)
compact_rows = []
for k in K_VALUES:
    m = KMeans(n_clusters=k, n_init=10, random_state=SEED)
    l = m.fit_predict(compact_X)
    compact_rows.append({'k': k, 'Silhouette': silhouette_score(compact_X, l,
        metric='euclidean', sample_size=1200, random_state=SEED), 'Smallest cluster': np.bincount(l).min()})
compact_results = pd.DataFrame(compact_rows)
print(compact_results.round(4).to_string(index=False))
print('Compare the best compact silhouette with the original 19-variable result, but do not infer that '
      'a higher score automatically means better financial insights.')

## 7. Describe the selected groups without leaking outcomes into clustering
Now that the clusters are fitted, I will assign each movie a cluster label and describe its features. This is the part I consider most useful for the capstone question: Do clusters capture recognizable types of films, such as high-budget action movies or smaller horror films?

I will report each group's median budget, median runtime, average release year, animation and genre percentages, and sample size. Then I will examine realized revenue and the two-times-budget success share. Those outcome statistics are **post-hoc descriptive comparisons** only. They were not used in feature scaling, K-Means fitting, or selecting k.

In [ ]:
train_profile = train.copy()
train_profile['cluster'] = labels
summary = train_profile.groupby('cluster').agg(
    movies=('movie_id','size'), median_budget_m=('budget_m','median'),
    median_runtime=('runtime','median'), mean_release_year=('release_year','mean'),
    animation_share=('animation','mean'), action_share=('genre_action','mean'),
    comedy_share=('genre_comedy','mean'), horror_share=('genre_horror','mean'),
    median_revenue_m=('revenue_m','median'), success_share=('success','mean'),
    median_gross_to_budget=('gross_to_budget','median')
).round(3)
print(summary.to_string())
print('Overall train success share:', round(train.success.mean(), 3))
print('These descriptive outcomes were NOT used when generating the clusters.')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
cluster_counts = summary['movies']
axs[0].bar(cluster_counts.index.astype(str), cluster_counts.values)
axs[0].set(title='Training movies in each cluster', xlabel='Cluster', ylabel='Movies')
axs[1].bar(summary.index.astype(str), summary['success_share'])
axs[1].axhline(train.success.mean(), linestyle='--', label='Training sample average')
axs[1].set(title='Observed financial success by cluster', xlabel='Cluster', ylabel='Share earning at least 2x budget', ylim=(0,1))
axs[1].legend(); plt.tight_layout(); plt.show()

## 8. PCA plot: a two-dimensional illustration, not proof of separation
The 19-variable clustering takes place in a much higher-dimensional space. I will use principal component analysis to create a two-dimensional **visualization** only. This projection may hide overlap or separation visible in the original space, and it should not replace silhouette scores or the numerical profiles.

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
xy = pca.fit_transform(X_final)
rng = np.random.default_rng(SEED)
idx = rng.choice(len(xy), size=min(2000, len(xy)), replace=False)
fig, ax = plt.subplots(figsize=(8, 5.2))
sc = ax.scatter(xy[idx,0], xy[idx,1], c=labels[idx], s=10, alpha=.6, cmap='tab10')
ax.set(title='PCA visualization of selected training clusters', xlabel='Principal component 1', ylabel='Principal component 2')
plt.colorbar(sc, ax=ax, label='Cluster ID'); plt.tight_layout(); plt.show()
print('Variance explained by two PCs:', round(pca.explained_variance_ratio_.sum(),4))

## 9. Stability and held-out assignment check
K-Means can produce different clusters depending on initial centroids, even with the same scaling and k. I will fit the selected configuration with several seeds on the **same training sample** and use adjusted Rand index (ARI) to compare the resulting labels. ARI of 1 means identical partitions (up to label renaming); values closer to 0 indicate little agreement beyond chance. A good silhouette without stability would make my clusters less convincing.

I will then assign the held-out test movies to the already-fitted centers and compare their cluster profiles. I will not refit cluster centers on test data. The train/test split is random, so this is a consistency check on similar movies, **not** a future-release validation.

In [ ]:
other_labels = []
for seed in [7, 11, 23, 42, 99]:
    m = KMeans(n_clusters=best_k, n_init=10, random_state=seed)
    other_labels.append(m.fit_predict(X_final))
ari = [adjusted_rand_score(labels, candidate) for candidate in other_labels]
print(pd.DataFrame({'Random seed': [7,11,23,42,99], 'ARI vs selected solution': ari}).round(4).to_string(index=False))

test_labels = best_model.predict(scaled_test[best_scaling])
test_profile = test.copy()
test_profile['cluster'] = test_labels
comparison = pd.concat([
    train_profile.groupby('cluster').agg(train_n=('movie_id','size'),train_success=('success','mean')),
    test_profile.groupby('cluster').agg(test_n=('movie_id','size'),test_success=('success','mean'))
], axis=1)
print(comparison.round(3).to_string())
print('Note: outcomes are descriptive checks only; they did not influence the unsupervised fit.')

## 10. Conclusions for my capstone and Milestone Two
This week allowed me to look at the film data from a different angle. Instead of asking a model to predict success directly, I am asking whether movies naturally group together based on characteristics that are known before release. The elbow plot and silhouette scores will show **whether the grouping is geometrically convincing**, while the profiles will show **whether it is meaningful from a movie-industry perspective**.

**Expected:** I expect budget and genre to affect the grouping, because high-budget action movies and lower-budget horror or comedy films can have very different characteristics. However, I should only write this as a finding if the executed cluster profiles actually support it.

**Unexpected / limitations to assess:** The best silhouette score might come from unscaled data dominated by budget or year, or a grouping that isolates only a small number of movies. Either case could be mathematically neat without giving useful business insights. I also expect PCA to reveal some overlap because genre, release month, and budget interact in complicated ways. My plots and results will determine whether this expectation holds.

**How I will justify the conclusions:** I will cite the selected k, the elbow curve, silhouette scores under multiple scalings, the Manhattan-distance diagnostic, cluster sizes, ARI stability, and post-hoc outcomes. I will distinguish a group of similar movies from a dependable prediction of box-office success. I will also explain that financial success at two times budget is only a proxy, not net profitability, and that this dataset excludes films without positive reported revenue or budgets.

**Paper-ready checklist (fill in after running the notebook):**
1. Which scaling method and k achieved the highest sampled Euclidean silhouette, and was the elbow consistent?
2. Were any clusters too small or dominated by one feature or indicator?
3. Did Manhattan versus Euclidean evaluation materially change the score?
4. Did the compact month encoding change the pattern?
5. Were the movie profiles substantively different, and were those differences stable on the held-out split?
6. What do the cluster differences actually add to the main research question?

**Interpretation rule:** I will not write numerical results for these checks until I have executed the notebook with my original `movies.csv`.